In [ ]:
import pandas as pd
import numpy as np
import re
from typing import List, Union
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import TextVectorization

from wordcloud import WordCloud
from matplotlib.colors import  ListedColormap
import matplotlib.pyplot as plt

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
print(tf.__version__)

2.17.0


In [ ]:
path = "/content/Marvel Vs DC.csv"
data = pd.read_csv(path)

In [ ]:
data.drop(["ID", "Metascore", "USA_Gross"], axis=1, inplace=True)
data.dropna(inplace=True, axis=0)

In [ ]:
def extract_year(data, column):
        data[column] = data[column].apply(
            lambda x: re.sub(r'^.*?(\d{4}).*', r'\1', x) if isinstance(x, str) else x)
        return data[column]

In [ ]:
data["Year"] = extract_year(data, "Year")
data["Year"] = pd.to_datetime(data["Year"], format="%Y")
data["Year"] = data["Year"].dt.year

data["Votes"] = pd.to_numeric(data['Votes'].str.replace(',', ''), errors='coerce')

data["Director"] = data["Director"].str.split(",")
data["Director"] = data["Director"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)

data = data.reset_index(drop=True)

In [ ]:
for _ in range(5):
  i = np.random.randint(len(data) - 1)
  print(f"Movie: {data['Movie'][i]}")
  print(data['Description'][i])
  print()

Movie: Being Flynn
Working in a Boston homeless shelter, Nick Flynn re-encounters his father, a con man and self-proclaimed poet. Sensing trouble in his own life, Nick wrestles with the notion of reaching out yet again to his dad.

Movie:  The Zeta Project
Zee and Ro go to a museum where Dr. Selig is giving a lecture.

Movie:  Gotham
On the precipice of Gotham's reunification, Gordon and Bruce find themselves face-to-face with a newly transformed Eduardo, and discover the real mastermind behind the city's current chaos. Meanwhile, a pregnant Barbara turns to Lee for help.

Movie:  Krypto the Superdog
While Andrea prepares to celebrate Hanukkah, Ignatius inadvertantly brings a huge, angry snowman to life. Krypto manages to defeat the angry snowmen just in time for everyone to celebrate Hanukkah.

Movie:  DC's Legends of Tomorrow
When the Legends discover a new Encore, Sara, Rory and Ava end up crashing a high school reunion to attempt to stop a serial killer from finishing unfinished bu

In [ ]:
def recomender_movies(category, genre, dataframe=data, top=10):
  movies = data[data["Category"] == category]
  movies = movies[movies["Genre"].str.contains(genre)]
  movies = movies.sort_values(by="Votes", ascending=False)
  movies = movies.reset_index(drop=True)
  movies.drop(["Category", "Genre", "Rating"], axis=1, inplace=True)
  movies.drop_duplicates(subset=['Movie'], inplace=True)
  return movies.head(top)

In [ ]:
category = "DC"
genre = "Animation"
recomender_movies(category, genre)

In [ ]:
def clean_text(text):
  text = text.lower()
  text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
  text = re.sub(r'\d+', '', text)
  text = re.sub(r'\s+', ' ', text)
  stop_words = set(stopwords.words('english'))
  text = ' '.join([word for word in text.split() if word not in stop_words])
  return text

In [ ]:
text = data['Description'][0]
print(text)
print(clean_text(text))

After the devastating events of Avengers: Infinity War (2018), the universe is in ruins. With the help of remaining allies, the Avengers assemble once more in order to reverse Thanos' actions and restore balance to the universe.
devastating events avengers infinity war universe ruins help remaining allies avengers assemble order reverse thanos actions restore balance universe


In [ ]:
data['Description'] = data['Description'].apply(clean_text)

In [ ]:
data_mavel = data[data["Category"] == "Marvel"].reset_index(drop=True)

data_dc = data[data["Category"] == "DC"].reset_index(drop=True)

In [ ]:
color_map = ListedColormap(['orange', 'green', 'red', 'magenta'])

cloud = WordCloud(background_color='white', max_words=100, colormap=color_map)
cloud = cloud.generate(data_mavel['Description'].str.cat(sep='\n'))
plt.figure(figsize=(15, 15))
plt.imshow(cloud)
plt.axis('off')
plt.show()

In [ ]:
color_map = ListedColormap(['orange', 'green', 'red', 'magenta'])

cloud = WordCloud(background_color='white', max_words=100, colormap=color_map)
cloud = cloud.generate(data_dc['Description'].str.cat(sep='\n'))
plt.figure(figsize=(15, 15))
plt.imshow(cloud)
plt.axis('off')
plt.show()

In [ ]:
descriptions = data["Description"].values
score = data["IMDB_Score"].values
score = score.reshape(-1, 1)

In [ ]:
max_len = max(len(description.split()) for description in descriptions)

vectorizer = TextVectorization(output_mode='int',split='whitespace',
                               standardize='strip_punctuation',max_tokens=10000,
                               output_sequence_length=max_len)
vectorizer.adapt(descriptions)
print(f'Vocabulary size: {vectorizer.vocabulary_size()}')

Vocabulary size: 5519


In [ ]:
for i in range(3):
  print(descriptions[i])
  print(vectorizer(descriptions[i]))
  print()

devastating events avengers infinity war universe ruins help remaining allies avengers assemble order reverse thanos actions restore balance universe
tf.Tensor(
[1534  284  717 4172  104  328 3250    9 1331  421  717 2491  102 3306
 1715  371  621  715  328    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0], shape=(37,), dtype=int64)

group intergalactic criminals must pull together stop fanatical warrior plans purge universe
tf.Tensor(
[ 119  549  885   10 1023   65   14 1116 1217   93 3454  328    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0], shape=(37,), dtype=int64)

following events avengers endgame spiderman must step take new threats world changed forever
tf.Tensor(
[ 406  284  717 1503 1777   10  215   42    8 1710   23  700  674    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0], shape=(

In [ ]:
def generate_dataset(descriptions_vectorized, score, buffer_size, batch_size, p):
    # Calculates the split size between training and testing
    size = int(p * len(descriptions_vectorized))
    print(f"Tamanho do conjunto de treino: {size}")

    # Creates the dataset from features and scores
    dataset = tf.data.Dataset.from_tensor_slices((descriptions_vectorized, score))

    dataset = dataset.shuffle(buffer_size=buffer_size)

    # Split the dataset into training and testing
    dataset_train = dataset.take(size)
    dataset_test = dataset.skip(size)

    # Apply batch and prefetch after split
    dataset_train = dataset_train.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    dataset_test = dataset_test.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return dataset_train, dataset_test

In [ ]:
p = .8
buffer_size = 10000
batch_size = 64
descriptions_vectorized = vectorizer(descriptions)
data_train, data_test = generate_dataset(descriptions_vectorized, score,
                                         buffer_size, batch_size, p)

Tamanho do conjunto de treino: 997


In [ ]:
class Model(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, rnn_units):

        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim,
                                                   name="Embedding")
        self.gru = tf.keras.layers.GRU(rnn_units,
                                       return_sequences=True,
                                       return_state=True,
                                       name="GRU")
        self.dense = tf.keras.layers.Dense(1, name="Dense")

    def call(self, inputs, states=None, return_state=False, training=False):
        x = inputs
        x = self.embedding(x, training=training)

        # Get the batch size from the input shape
        batch_size = tf.shape(x)[0]

        if states is None:
            states = self.gru.get_initial_state(batch_size=batch_size)
        x, states = self.gru(x, initial_state=states, training=training)
        x = self.dense(x, training=training)

        if return_state:
            return x, states
        else:
            return x

In [ ]:
vocab_size = vectorizer.vocabulary_size()
embedding_dim = 128
rnn_units = 128

model = Model(vocab_size, embedding_dim, rnn_units)

# Use a valid batch size when building the model.
model.build(input_shape=(batch_size, 100))

# Use a non-zero batch size for the input tensor
input_tensor = tf.constant(tf.zeros(shape=(batch_size, 100)))

# Call the model with the concrete tensor
model.call(input_tensor)
model.summary()

/usr/local/lib/python3.10/dist-packages/keras/src/layers/layer.py:372: UserWarning: `build()` was called on layer 'model', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ Embedding (Embedding)                │ (64, 100, 128)              │         706,432 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ GRU (GRU)                            │ ((64, 100, 128), (64, 128)) │          99,072 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Dense (Dense)                        │ (64, 100, 1)                │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 805,633 (3.07 MB)

 Trainable params: 805,633 (3.07 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
def compile_model(model: tf.keras.Model,
                  optimizer: tf.keras.optimizers.Optimizer = None,
                  loss: Union[tf.keras.losses.Loss, str, None] = None,
                  metrics: Union[List[Union[tf.keras.metrics.Metric, str]], None] = None,
                  learning_rate: float = 0.001) -> tf.keras.Model:

    # Sets the default optimizer if not provided
    if optimizer is None:
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    # Sets the default loss function if not provided
    if loss is None:
        loss = tf.keras.losses.Huber()

    # Sets default metrics if not provided
    if metrics is None:
        metrics=['mae', 'mse']

    # Compile the model
    model.compile(optimizer=optimizer,
                  loss=loss,
                  metrics=metrics)

    return model

In [ ]:
def my_callbacks(patience: int = 5, verbose: int = 1) -> List[tf.keras.callbacks.Callback]:
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=verbose)

    # You can add other callbacks if you want, like ModelCheckpoint, ReduceLROnPlateau etc.
    # For example, to save the best model during training:
    model_checkpoint = tf.keras.callbacks.ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_loss', verbose=verbose)

    return [early_stopping, model_checkpoint]

In [ ]:
def train_model(model: tf.keras.Model,
                data_train: tf.data.Dataset,
                data_test: tf.data.Dataset,
                callbacks: List[tf.keras.callbacks.Callback],
                epochs: int = 10,
                verbose: int = 1) -> tf.keras.callbacks.History:

    history = model.fit(data_train,
                        epochs=epochs,
                        verbose=verbose,
                        validation_data=data_test,
                        callbacks=callbacks)

    return history

In [ ]:
model = compile_model(model)
mycallback = my_callbacks()

# Treine o modelo
history = train_model(model, data_train, data_test, mycallback, epochs=15)

Epoch 1/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - loss: 6.0745 - mae: 6.5702 - mse: 48.0133
Epoch 1: val_loss improved from inf to 3.38790, saving model to best_model.keras
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 166ms/step - loss: 6.0092 - mae: 6.5047 - mse: 47.3689 - val_loss: 3.3879 - val_mae: 3.8667 - val_mse: 23.1337
Epoch 2/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step - loss: 3.1023 - mae: 3.5488 - mse: 21.4347
Epoch 2: val_loss improved from 3.38790 to 2.38092, saving model to best_model.keras
16/16 ━━━━━━━━━━━━━━━━━━━━ 6s 252ms/step - loss: 3.0918 - mae: 3.5375 - mse: 21.3460 - val_loss: 2.3809 - val_mae: 2.8039 - val_mse: 13.7883
Epoch 3/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - loss: 1.8681 - mae: 2.2863 - mse: 10.0932
Epoch 3: val_loss improved from 2.38092 to 0.77807, saving model to best_model.keras
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 109ms/step - loss: 1.8433 - mae: 2.2605 - mse: 9.9387 - val_loss: 0.7781 - val_mae: 1.1435 - val_mse: 3.5409
Epoch 4/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [ ]:
def plot_history(history, metric):
    plt.plot(history.history[metric])
    plt.plot(history.history['val_' + metric])
    plt.xlabel("Epochs")
    plt.ylabel(metric)
    plt.legend([metric, 'val_' + metric])
    plt.show()

In [ ]:
plot_history(history, 'loss')
plot_history(history, 'mae')
plot_history(history, 'mse')